# Experiment 3: Regression Analysis using Linear and Regularized Models
**Course:** ICS1512 - Machine Learning Algorithms Laboratory  
**Student Name:** Rishi Rithesh  
**Register Number:** 3122247001049

## 1. Objective
To implement and compare Linear Regression, Ridge Regression, Lasso Regression, and Elastic Net Regression for predicting loan amounts. The experiment involves performing exploratory data analysis, handling missing values, encoding categorical features, scaling numerical features, tuning hyperparameters using 5-fold GridSearchCV, and evaluating model performance using MAE, MSE, RMSE, R²-score, learning curves, and training time.

## 2. Import Libraries & Set Up Environment

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

## 3. Load Dataset & Correlation Matrix EDA

In [ ]:
csv_path = 'loan_amount_prediction_uncleaned_dataset.csv'
df_raw = pd.read_csv(csv_path)

# Correlation matrix of numerical features
num_df = df_raw.select_dtypes(include=['int64', 'float64']).dropna()
corr = num_df.corr()

plt.figure(figsize=(10, 8))
plt.matshow(corr, fignum=1, cmap='coolwarm')
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Matrix Heatmap", pad=20)
plt.show()

## 4. Data Preprocessing & Target Cleaning

In [ ]:
df = df_raw.dropna(subset=['loan_amount']).copy()
X = df.drop(columns=['loan_amount'])
y = df['loan_amount']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

## 5. Model Training & Hyperparameter Tuning (5-Fold CV)

In [ ]:
models = {
    'Linear Regression': (LinearRegression(), {}),
    'Ridge Regression': (Ridge(random_state=42), {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}),
    'Lasso Regression': (Lasso(random_state=42, max_iter=20000), {'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0]}),
    'Elastic Net': (ElasticNet(random_state=42, max_iter=20000), {'model__alpha': [0.01, 0.1, 1.0, 10.0], 'model__l1_ratio': [0.2, 0.5, 0.8]})
}

best_models = {}
test_results = []

for name, (model_obj, param_grid) in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model_obj)
    ])
    start = time.time()
    if param_grid:
        grid = GridSearchCV(pipe, param_grid, cv=5, scoring='r2', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_pipe = grid.best_estimator_
        print(f"[{name}] Best Params: {grid.best_params_} | Best CV R2: {grid.best_score_:.4f}")
    else:
        best_pipe = pipe
        best_pipe.fit(X_train, y_train)
        print(f"[{name}] Baseline Linear Regression Trained")
    
    t_time = time.time() - start
    best_models[name] = best_pipe
    
    y_pred = best_pipe.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    test_results.append({
        'Model': name,
        'MAE': round(mae, 2),
        'MSE': f"{mse:.3e}",
        'RMSE': round(rmse, 2),
        'R2 Score': round(r2, 4),
        'Training Time (s)': round(t_time, 4)
    })

pd.DataFrame(test_results)